# 04 LeRobot PushT环境与Policy推理闭环

目标：真正跑通一个具身智能推理闭环。

本Notebook完成：

1. 安装 PushT 环境
2. 阅读 PushT 源码结构
3. 加载 ACT Policy
4. 使用预训练模型推理
5. 分析 Policy 输入输出动作

最终理解：

```
observation
      ↓
 Policy(ACT)
      ↓
 action
      ↓
 PushT Environment
      ↓
 next observation
```



# 1. PushT是什么？

PushT 是 LeRobot 中经典模仿学习任务。

任务：

机器人控制一个圆形末端执行器，把 T 形方块推动到目标区域。

数据集：

```
lerobot/pusht
```

包含：

- observation.image
- observation.state
- action
- episode

但是：

数据集 ≠ 环境。

数据集保存过去轨迹。

环境负责：

```
s(t+1)=f(s(t),a(t))
```



In [ ]:
# 环境检查

import torch
import lerobot

print(torch.__version__)
print("lerobot OK")


2.11.0+cu128
lerobot OK


# 2. 安装PushT环境

LeRobot中的PushT环境来自机器人benchmark环境。

安装：

```
pip install gymnasium
pip install pygame
pip install pymunk
pip install gym-pusht
```

说明：

PushT不是MuJoCo任务，而是一个2D物理推送环境。

它使用：

- pygame显示
- pymunk物理引擎

因此：

MuJoCo和PushT都是Environment，但物理实现不同。


In [ ]:
# 尝试导入PushT

try:
    import gym_pusht
    print("PushT environment installed")
except Exception as e:
    print(e)


PushT environment installed


# 3. 查看PushT源码

安装后查看：

```
site-packages/gym_pusht
```

主要结构：

```
gym_pusht
|
├── envs
|    |
|    └── pusht.py
|
├── assets
|
└── utils
```

重点：

环境类一般包含：

```
reset()
step(action)
render()
```



In [ ]:
# 查看安装位置

import gym_pusht
print(gym_pusht.__file__)


d:\Desktop\robot\envs\lerobot-win\Lib\site-packages\gym_pusht\__init__.py


# 4. 创建PushT环境

Gymnasium统一接口：

```
env.reset()

env.step(action)
```



In [ ]:
import gymnasium as gym

# 示例
# env = gym.make("PushT-v0")

print("环境创建代码示例")


环境创建代码示例


# 5. LeRobot Dataset和PushT环境关系

训练阶段：

```
lerobot/pusht dataset

observation + action

        ↓

训练ACT Policy
```


测试阶段：

```
PushT Environment

observation

        ↓

ACT Policy

        ↓

action

        ↓

env.step(action)
```



# 6. 加载ACT Policy

ACT:

Action Chunking Transformer

输入：

```
image
robot state
```

输出：

```
未来多个动作

[action_t,...,action_t+N]
```

LeRobot提供预训练模型。

示例：

```
lerobot/act_pusht
```



In [ ]:
# ACT加载示意

# from lerobot.policies.act.modeling_act import ACTPolicy

# policy = ACTPolicy.from_pretrained(
#     "lerobot/act_pusht"
# )

print("ACT Policy loading example")


# 7. Policy输入输出分析

Policy输入：

```
observation

{
 image:
 Tensor(3,H,W),

 state:
 Tensor(n)
}
```


输出：

```
action chunk

Tensor(T, action_dim)
```


例如：

```
[
 [x1,y1],
 [x2,y2],
 [x3,y3]
]
```

表示未来连续动作。


In [ ]:
# 动作结构示例

import torch

action_chunk = torch.randn(10,2)

print(action_chunk.shape)
print(action_chunk)


# 8. 完整推理闭环


伪代码：


```
obs = env.reset()

while True:

    action = policy(obs)

    obs,reward,done,info = env.step(action)

```


这里：

Policy负责：

```
决定怎么做
```

Environment负责：

```
世界如何变化
```



# 9. 分析关键概念

## 为什么训练数据不包含f？

因为：

dataset:

```
过去发生什么
```

environment:

```
未来如何变化
```


## 为什么需要环境？

因为模型最终需要测试：

```
动作是否真的有效
```



# 10. 后续学习

完成04后：

05_ACT源码解析

学习：

- Transformer结构
- Encoder输入
- Decoder输出
- action chunking
- loss
- training loop

06_Diffusion Policy

学习：

- diffusion生成动作
- noise schedule
- denoising policy

